In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
from tqdm import tqdm
import json
import numpy as np
import ast

from neo4j import GraphDatabase

In [ ]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")


with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [75]:
cypher = "MATCH (n) RETURN n"
records, summary, keys = driver.execute_query(cypher)
records

[<Record n=<Node element_id='4:8197a03e-5e78-47d7-885e-79bf1e759e75:0' labels=frozenset({'Claim'}) properties={'locution': 'Even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value', 'prediction_index': 0, 'utterance_type': '___Claim___', 'ensemble1_political_position_array': [30, 30, 30, 30, 30, 35, 30, 30, 30, 50, 50, 50, 50, 50, 30, 30, 50, 50, 20, 50, 20, 50, 50, 80, 50, 80, 50, 50, 20, 20, 20, 20, 40, 90, 90, 90, 90, 90, 50, 50, 50, 40, 50, 40, 40, 40, 40, 40, 60, 60, 60, 60, 60, 40, 40, 50, 40, 40], 'ensemble1_political_position_na_count': 52, 'ensemble2_political_position_mean': 32.8125, 'ensemble3_political_position_probability_of_na': 0.7333333333333333, 'ensemble1_political_position_probability_of_na': 0.4727272727272727, 'ensemble3_political_position_na_count': 44, 'ensemble2_political_position_array': [30, 30, 30, 30, 30, 35, 30, 30, 

In [76]:
len(records)

23228

# Ensemble 1: All Results

In [ ]:
all_models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_35_haiku.tsv","anthropic/claude_37_sonnet.tsv", 
          "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
          "openai/gpt35_turbo.tsv", "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt4o_mini.tsv", "openai/o3_mini.tsv", "openai/gpt45_preview.tsv",
          "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
          "xai/grok2.tsv", 
          "meta/llama31_8b.tsv", "meta/llama31_405b.tsv", "meta/llama32_3b.tsv", "meta/llama33_70b.tsv", "meta/llama4_maverick.tsv",
          "mistral/mistral_7b.tsv",
          "microsoft/phi4.tsv"]
all_model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
               "Claude 3.5 Haiku", "Claude 3.7 Sonnet", 
               "DeepSeek-V3", "DeepSeek-R1",
               "GPT 3.5 Turbo", "GPT 4o", "GPT 4o Turbo", "GPT 4o Mini", "o3 Mini", "GPT 4.5 Preview",
               "Gemini 1.5 Pro", "Gemini 2.5 Flash",
               "Grok 2", 
               "Llama 3.1:8b", "Llama 3.1:405b", "Llama 3.2:3b", "Llama 3.3:70b", "Llama 4 Maverick",
               "Mistral:7b",
               "Phi 4"]

In [ ]:
ensemble1_df = pd.read_csv("../../data/processed/PoliticalPositions/ensembles/ensemble1.tsv", delimiter="\t")

In [48]:
ensemble1_df.head()

,nodeID,array_of_pol_pos,mean,var,std,NA_count,prob_of_NA
0,0,"[30, 30, 30, 30, 30, 35, 30, 30, 30, 50, 50, ...",46.637931,351.196492,18.740237,52,0.472727
1,1,"[50, 50, 50, 50, 50, 65, 50, 50, 20, 50, 50, ...",42.027027,328.999270,18.138337,73,0.663636
2,2,"[50, 50, 50, 50, 50, 75, 75, 50, 50, 50, 50, ...",43.750000,390.625000,19.764235,78,0.709091
3,3,"[30, 40, 50, 50, 50, 50, 50, 30, 25, 40, 40, ...",39.857143,306.408163,17.504518,75,0.681818
4,4,"[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, ...",44.743590,397.370151,19.934145,71,0.645455


In [ ]:
data_base_connection = GraphDatabase.driver(uri = "bolt://localhost:7687", auth=("neo4j", "password"))
session = data_base_connection.session()  

In [70]:
ensemble_string = "ensemble1"

for record in tqdm(records):
    #print(record["n"]._properties["prediction_index"])
    i = record["n"]._properties["prediction_index"]

    #print(ensemble1_df.iloc[i])
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "])
    #print(type(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0]))
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0])
    political_position_array = ast.literal_eval(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0].strip())
    #print(type(political_position_array))
    political_position_mean = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" mean "].values[0]
    political_position_std = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" std "].values[0]
    political_position_na_count = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" NA_count "].values[0]
    political_position_probability_of_na = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" prob_of_NA "].values[0]

    cypher = f'''MATCH (n) WHERE n.prediction_index = {i} 
    SET n.{ensemble_string}_political_position_array = {political_position_array}, n.{ensemble_string}_political_position_mean = {political_position_mean}, n.{ensemble_string}_political_position_std = {political_position_std}, n.{ensemble_string}_political_position_na_count = {political_position_na_count}, n.{ensemble_string}_political_position_probability_of_na = {political_position_probability_of_na}'''

    session.run(cypher)
    

100%|██████████| 23228/23228 [03:23<00:00, 114.25it/s]


# Ensemble 2: Reasoning Models

In [ ]:
reasoning_models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_37_sonnet.tsv", 
           "deepseek/deepseek_r1.tsv",
          "openai/o3_mini.tsv"]
reasoning_model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
                "Claude 3.7 Sonnet", 
               "DeepSeek-R1",
               "o3 Mini"]

In [71]:
ensemble1_df = pd.read_csv("../../data/processed/PoliticalPositions/ensembles/ensemble2.tsv", delimiter="\t")

ensemble_string = "ensemble2"

for record in tqdm(records):
    #print(record["n"]._properties["prediction_index"])
    i = record["n"]._properties["prediction_index"]

    #print(ensemble1_df.iloc[i])
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "])
    #print(type(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0]))
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0])
    political_position_array = ast.literal_eval(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0].strip())
    #print(type(political_position_array))
    political_position_mean = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" mean "].values[0]
    political_position_std = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" std "].values[0]
    political_position_na_count = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" NA_count "].values[0]
    political_position_probability_of_na = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" prob_of_NA "].values[0]

    cypher = f'''MATCH (n) WHERE n.prediction_index = {i} 
    SET n.{ensemble_string}_political_position_array = {political_position_array}, n.{ensemble_string}_political_position_mean = {political_position_mean}, n.{ensemble_string}_political_position_std = {political_position_std}, n.{ensemble_string}_political_position_na_count = {political_position_na_count}, n.{ensemble_string}_political_position_probability_of_na = {political_position_probability_of_na}'''

    session.run(cypher)
    

100%|██████████| 23228/23228 [02:54<00:00, 133.09it/s]


# Ensemble 3: Models where |Predictions| > Number of NA scores

In [ ]:
ensemble3_models = ["alibaba_cloud/qwen3_235b.tsv",
              "anthropic/claude_37_sonnet.tsv", 
              "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
              "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt45_preview.tsv",
              "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
              "xai/grok2.tsv", 
              "meta/llama4_maverick.tsv",
              "microsoft/phi4.tsv"]
ensemble3_model_names = ["Qwen 3 235b",
                   "Claude 3.7 Sonnet", 
                   "DeepSeek-V3", "DeepSeek-R1",
                   "GPT 4o", "GPT 4 Turbo", "GPT 4.5 Preview",
                   "Gemini 1.5 Pro", "Gemini 2.5 Flash",
                   "Grok 2", 
                   "Llama 4 Maverick",
                   "Phi 4"]

In [72]:
ensemble1_df = pd.read_csv("../../data/processed/PoliticalPositions/ensembles/ensemble3.tsv", delimiter="\t")

ensemble_string = "ensemble3"

for record in tqdm(records):
    #print(record["n"]._properties["prediction_index"])
    i = record["n"]._properties["prediction_index"]

    #print(ensemble1_df.iloc[i])
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "])
    #print(type(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0]))
    #print(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0])
    political_position_array = ast.literal_eval(ensemble1_df.loc[ensemble1_df["nodeID "] == i][" array_of_pol_pos "].values[0].strip())
    #print(type(political_position_array))
    political_position_mean = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" mean "].values[0]
    political_position_std = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" std "].values[0]
    political_position_na_count = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" NA_count "].values[0]
    political_position_probability_of_na = ensemble1_df.loc[ensemble1_df["nodeID "] == i][" prob_of_NA "].values[0]

    cypher = f'''MATCH (n) WHERE n.prediction_index = {i} 
    SET n.{ensemble_string}_political_position_array = {political_position_array}, n.{ensemble_string}_political_position_mean = {political_position_mean}, n.{ensemble_string}_political_position_std = {political_position_std}, n.{ensemble_string}_political_position_na_count = {political_position_na_count}, n.{ensemble_string}_political_position_probability_of_na = {political_position_probability_of_na}'''

    session.run(cypher)
    

100%|██████████| 23228/23228 [03:03<00:00, 126.71it/s]


# Adding individual models

In [80]:
all_models_paths = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_35_haiku.tsv","anthropic/claude_37_sonnet.tsv", 
          "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
          "openai/gpt35_turbo.tsv", "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt4o_mini.tsv", "openai/o3_mini.tsv", "openai/gpt45_preview.tsv",
          "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
          "xai/grok2.tsv", 
          "meta/llama31_8b.tsv", "meta/llama31_405b.tsv", "meta/llama32_3b.tsv", "meta/llama33_70b.tsv", "meta/llama4_maverick.tsv",
          "mistral/mistral_7b.tsv",
          "microsoft/phi4.tsv"]

In [81]:

all_model_names = ["qwen3_235b_a22b", "qwen_qwq_32b",
               "claude_3_5_haiku_2024102", "claude_3_7_sonnet_20250219", 
               "deepseek_v3", "deepseek_r1",
               "gpt_3_5_turbo_0125", "gpt_4o_2024_08_06", "gpt_4_turbo_2024_04_09", "gpt_4o_mini_2024_07_18", "o3_mini_2025_01_31", "gpt_4_5_preview_2025_02_27",
               "gemini_1_5_pro_002", "gemini_2_5_flash_preview",
               "grok_2_1212", 
               "llama3_1_8b", "llama_3_1_405b_instruct", "llama3_2_3b", "llama_3_3_70b_instruct", "llama_4_maverick",
               "mistral_7b",
               "phi_4"]

In [82]:
for model, file_path in tqdm(zip(all_model_names, all_models_paths)):
    df = pd.read_csv(f"../../data/processed/PoliticalPositions/models/{file_path}", delimiter="\t")


    for record in records:
        #print(record["n"]._properties["prediction_index"])
        i = record["n"]._properties["prediction_index"]

        political_position_array = ast.literal_eval(df.loc[df["nodeID "] == i][" array_of_pol_pos "].values[0].strip())
        #print(type(political_position_array))
        political_position_mean = df.loc[df["nodeID "] == i][" mean "].values[0]
        political_position_std = df.loc[df["nodeID "] == i][" std "].values[0]
        political_position_na_count = df.loc[df["nodeID "] == i][" NA_count "].values[0]
        political_position_probability_of_na = df.loc[df["nodeID "] == i][" prob_of_NA "].values[0]

        cypher = f'''MATCH (n) WHERE n.prediction_index = {i} 
        SET n.{model}_political_position_array = {political_position_array}, n.{model}_political_position_mean = {political_position_mean}, n.{model}_political_position_std = {political_position_std}, n.{model}_political_position_na_count = {political_position_na_count}, n.{model}_political_position_probability_of_na = {political_position_probability_of_na}'''

        session.run(cypher)

22it [1:12:28, 197.68s/it]
